In [1]:
from pathlib import Path
import json
import sys
import datetime

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Fraud-detection-ML-V2 project root could not be found."
    )

SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
REPORTS_DIR = OUTPUTS_DIR / "reports"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT, SRC_DIR]:
    path_string = str(path)
    if path_string not in sys.path:
        sys.path.insert(0, path_string)

print("PHASE 15 PROJECT CHECK")
print("=" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Source       : {SRC_DIR}")
print(f"Models       : {MODELS_DIR}")
print(f"Metrics      : {METRICS_DIR}")
print(f"Reports      : {REPORTS_DIR}")
print("=" * 60)
print("Project discovery: PASS")

PHASE 15 PROJECT CHECK
Project root : C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2
Source       : C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src
Models       : C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models
Metrics      : C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics
Reports      : C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\reports
Project discovery: PASS


In [2]:
PERSON_A_TO_B_FIELDS = {
    "transaction_id": str,
    "user_id": str,
    "amount": float,
    "amount_vs_avg_ratio": float,
    "txn_count_last_5min": int,
    "time_since_last_txn_sec": float,
    "distance_from_last_location_km": float,
    "merchant_category_is_new_for_user": (bool, int)
}

PERSON_A_TO_B_FIELD_NAMES = list(
    PERSON_A_TO_B_FIELDS.keys()
)

print("PERSON A → PERSON B CONTRACT")
print("=" * 60)

for field_name in PERSON_A_TO_B_FIELD_NAMES:
    print(
        f"{field_name}: "
        f"{PERSON_A_TO_B_FIELDS[field_name]}"
    )

print("=" * 60)
print("Feature Vector contract defined: READY")

PERSON A → PERSON B CONTRACT
transaction_id: <class 'str'>
user_id: <class 'str'>
amount: <class 'float'>
amount_vs_avg_ratio: <class 'float'>
txn_count_last_5min: <class 'int'>
time_since_last_txn_sec: <class 'float'>
distance_from_last_location_km: <class 'float'>
merchant_category_is_new_for_user: (<class 'bool'>, <class 'int'>)
Feature Vector contract defined: READY


In [3]:
PERSON_B_TO_C_FIELDS = {
    "transaction_id": str,
    "user_id": str,
    "risk_score": (int, float),
    "rule_flags": list,
    "ml_fraud_score": (int, float),
    "decision": str,
    "human_readable_reason": str,
    "processed_at": str,
    "latency_ms": (int, float)
}

PERSON_B_TO_C_FIELD_NAMES = list(
    PERSON_B_TO_C_FIELDS.keys()
)

ALLOWED_DECISIONS = {
    "Allow",
    "OTP",
    "Review",
    "Block",
    "allow",
    "otp",
    "review",
    "block"
}

print("PERSON B → PERSON C CONTRACT")
print("=" * 60)

for field_name in PERSON_B_TO_C_FIELD_NAMES:
    print(
        f"{field_name}: "
        f"{PERSON_B_TO_C_FIELDS[field_name]}"
    )

print("=" * 60)
print("Scoring Output contract defined: READY")

PERSON B → PERSON C CONTRACT
transaction_id: <class 'str'>
user_id: <class 'str'>
risk_score: (<class 'int'>, <class 'float'>)
rule_flags: <class 'list'>
ml_fraud_score: (<class 'int'>, <class 'float'>)
decision: <class 'str'>
human_readable_reason: <class 'str'>
processed_at: <class 'str'>
latency_ms: (<class 'int'>, <class 'float'>)
Scoring Output contract defined: READY


In [4]:
feature_vector = {
    "transaction_id": "CONTRACT_TXN_000001",
    "user_id": "CONTRACT_USER_000001",
    "amount": 250.0,
    "amount_vs_avg_ratio": 1.5,
    "txn_count_last_5min": 1,
    "time_since_last_txn_sec": 300.0,
    "distance_from_last_location_km": 5.0,
    "merchant_category_is_new_for_user": 0
}

print("REPRESENTATIVE FEATURE VECTOR")
print("=" * 60)
print(json.dumps(feature_vector, indent=2))
print("=" * 60)

REPRESENTATIVE FEATURE VECTOR
{
  "transaction_id": "CONTRACT_TXN_000001",
  "user_id": "CONTRACT_USER_000001",
  "amount": 250.0,
  "amount_vs_avg_ratio": 1.5,
  "txn_count_last_5min": 1,
  "time_since_last_txn_sec": 300.0,
  "distance_from_last_location_km": 5.0,
  "merchant_category_is_new_for_user": 0
}


In [6]:
def validate_feature_vector(data):
    errors = []

    if not isinstance(data, dict):
        return ["Feature Vector must be a dictionary."]

    expected_fields = {
        "transaction_id",
        "user_id",
        "amount",
        "amount_vs_avg_ratio",
        "txn_count_last_5min",
        "time_since_last_txn_sec",
        "distance_from_last_location_km",
        "merchant_category_is_new_for_user"
    }

    actual_fields = set(data.keys())

    missing_fields = expected_fields - actual_fields
    extra_fields = actual_fields - expected_fields

    if missing_fields:
        errors.append(
            "Missing fields: "
            + ", ".join(sorted(missing_fields))
        )

    if extra_fields:
        errors.append(
            "Unexpected fields: "
            + ", ".join(sorted(extra_fields))
        )

    if "transaction_id" in data:
        if not isinstance(data["transaction_id"], str):
            errors.append(
                "transaction_id must be a string."
            )

    if "user_id" in data:
        if not isinstance(data["user_id"], str):
            errors.append(
                "user_id must be a string."
            )

    numeric_fields = {
        "amount",
        "amount_vs_avg_ratio",
        "time_since_last_txn_sec",
        "distance_from_last_location_km"
    }

    for field in numeric_fields:
        if field in data:
            value = data[field]

            if (
                not isinstance(value, (int, float))
                or isinstance(value, bool)
            ):
                errors.append(
                    f"{field} must be numeric."
                )

    if "txn_count_last_5min" in data:
        value = data["txn_count_last_5min"]

        if (
            not isinstance(value, int)
            or isinstance(value, bool)
        ):
            errors.append(
                "txn_count_last_5min must be an integer."
            )

    if "merchant_category_is_new_for_user" in data:
        value = data["merchant_category_is_new_for_user"]

        if not isinstance(value, (bool, int)):
            errors.append(
                "merchant_category_is_new_for_user must be bool or int."
            )

    return errors


feature_vector_errors = validate_feature_vector(
    feature_vector
)

print("FEATURE VECTOR VALIDATION")
print("=" * 60)

if feature_vector_errors:
    for error in feature_vector_errors:
        print(error)

    raise ValueError(
        "Feature Vector contract validation failed."
    )

print("Status: PASS")
print("Person A → Person B contract is valid.")

FEATURE VECTOR VALIDATION
Status: PASS
Person A → Person B contract is valid.


In [7]:
invalid_feature_vectors = {
    "missing_amount": {
        key: value
        for key, value in feature_vector.items()
        if key != "amount"
    },

    "missing_user_id": {
        key: value
        for key, value in feature_vector.items()
        if key != "user_id"
    },

    "wrong_amount_type": {
        **feature_vector,
        "amount": "250"
    },

    "wrong_ratio_type": {
        **feature_vector,
        "amount_vs_avg_ratio": "1.5"
    },

    "wrong_velocity_type": {
        **feature_vector,
        "txn_count_last_5min": "1"
    },

    "unexpected_field": {
        **feature_vector,
        "unexpected_field": "test"
    },

    "wrong_transaction_id_type": {
        **feature_vector,
        "transaction_id": 12345
    }
}

invalid_feature_vector_results = []

for test_name, test_data in invalid_feature_vectors.items():

    errors = validate_feature_vector(
        test_data
    )

    invalid_feature_vector_results.append({
        "test": test_name,
        "rejected": bool(errors),
        "errors": errors
    })

for result in invalid_feature_vector_results:
    print(
        f"{result['test']}: "
        f"{'REJECTED' if result['rejected'] else 'NOT REJECTED'}"
    )

unexpected_acceptance = [
    result
    for result in invalid_feature_vector_results
    if not result["rejected"]
]

if unexpected_acceptance:
    raise RuntimeError(
        "Some deliberately invalid Feature Vectors "
        "were not rejected."
    )

print("=" * 60)
print("Invalid Feature Vector testing: PASS")

missing_amount: REJECTED
missing_user_id: REJECTED
wrong_amount_type: REJECTED
wrong_ratio_type: REJECTED
wrong_velocity_type: REJECTED
unexpected_field: REJECTED
wrong_transaction_id_type: REJECTED
Invalid Feature Vector testing: PASS


In [8]:
mock_scoring_output = {
    "transaction_id": "CONTRACT_TXN_000001",
    "user_id": "CONTRACT_USER_000001",
    "risk_score": 0.32,
    "rule_flags": [],
    "ml_fraud_score": 0.41,
    "decision": "OTP",
    "human_readable_reason": "Transaction requires additional verification.",
    "processed_at": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "latency_ms": 5.2
}

print("MOCK SCORING OUTPUT")
print("=" * 60)
print(
    json.dumps(
        mock_scoring_output,
        indent=2
    )
)
print("=" * 60)

MOCK SCORING OUTPUT
{
  "transaction_id": "CONTRACT_TXN_000001",
  "user_id": "CONTRACT_USER_000001",
  "risk_score": 0.32,
  "rule_flags": [],
  "ml_fraud_score": 0.41,
  "decision": "OTP",
  "human_readable_reason": "Transaction requires additional verification.",
  "processed_at": "2026-09-06T00:49:31.147474+00:00",
  "latency_ms": 5.2
}


In [9]:
def validate_scoring_output(data):
    errors = []

    if not isinstance(data, dict):
        return [
            "Scoring Output must be a dictionary."
        ]

    expected_fields = set(
        PERSON_B_TO_C_FIELDS.keys()
    )

    actual_fields = set(
        data.keys()
    )

    missing_fields = expected_fields - actual_fields
    extra_fields = actual_fields - expected_fields

    if missing_fields:
        errors.append(
            "Missing fields: "
            + ", ".join(sorted(missing_fields))
        )

    if extra_fields:
        errors.append(
            "Unexpected fields: "
            + ", ".join(sorted(extra_fields))
        )

    if "transaction_id" in data:
        if not isinstance(
            data["transaction_id"],
            str
        ):
            errors.append(
                "transaction_id must be a string."
            )

    if "user_id" in data:
        if not isinstance(
            data["user_id"],
            str
        ):
            errors.append(
                "user_id must be a string."
            )

    for score_field in [
        "risk_score",
        "ml_fraud_score",
        "latency_ms"
    ]:
        if score_field in data:
            value = data[score_field]

            if not isinstance(
                value,
                (int, float)
            ) or isinstance(value, bool):
                errors.append(
                    f"{score_field} must be numeric."
                )

    if "risk_score" in data:
        value = data["risk_score"]

        if isinstance(value, (int, float)):
            if value < 0 or value > 1:
                errors.append(
                    "risk_score must be between 0 and 1."
                )

    if "ml_fraud_score" in data:
        value = data["ml_fraud_score"]

        if isinstance(value, (int, float)):
            if value < 0 or value > 1:
                errors.append(
                    "ml_fraud_score must be between 0 and 1."
                )

    if "rule_flags" in data:
        if not isinstance(
            data["rule_flags"],
            list
        ):
            errors.append(
                "rule_flags must be a list."
            )

    if "decision" in data:
        if data["decision"] not in ALLOWED_DECISIONS:
            errors.append(
                "decision contains an unsupported value."
            )

    if "human_readable_reason" in data:
        if not isinstance(
            data["human_readable_reason"],
            str
        ):
            errors.append(
                "human_readable_reason must be a string."
            )

    if "processed_at" in data:
        if not isinstance(
            data["processed_at"],
            str
        ):
            errors.append(
                "processed_at must be a string."
            )

    return errors

scoring_output_errors = validate_scoring_output(
    mock_scoring_output
)

print("SCORING OUTPUT VALIDATION")
print("=" * 60)

if scoring_output_errors:
    for error in scoring_output_errors:
        print(error)

    raise ValueError(
        "Scoring Output validation failed."
    )

print("Status: PASS")
print("Person B → Person C contract is valid.")

SCORING OUTPUT VALIDATION
Status: PASS
Person B → Person C contract is valid.


In [10]:
invalid_scoring_outputs = {
    "missing_risk_score": {
        key: value
        for key, value in mock_scoring_output.items()
        if key != "risk_score"
    },

    "invalid_risk_score": {
        **mock_scoring_output,
        "risk_score": 1.5
    },

    "invalid_ml_score": {
        **mock_scoring_output,
        "ml_fraud_score": -0.2
    },

    "invalid_rule_flags": {
        **mock_scoring_output,
        "rule_flags": "HIGH_AMOUNT"
    },

    "invalid_decision": {
        **mock_scoring_output,
        "decision": "UNKNOWN"
    },

    "invalid_latency": {
        **mock_scoring_output,
        "latency_ms": "5.2"
    },

    "invalid_user_id": {
        **mock_scoring_output,
        "user_id": 12345
    }
}

invalid_scoring_results = []

for test_name, test_data in invalid_scoring_outputs.items():

    errors = validate_scoring_output(
        test_data
    )

    invalid_scoring_results.append({
        "test": test_name,
        "rejected": bool(errors),
        "errors": errors
    })

for result in invalid_scoring_results:
    print(
        f"{result['test']}: "
        f"{'REJECTED' if result['rejected'] else 'NOT REJECTED'}"
    )

unexpected_acceptance = [
    result
    for result in invalid_scoring_results
    if not result["rejected"]
]

if unexpected_acceptance:
    raise RuntimeError(
        "Some deliberately invalid Scoring Outputs "
        "were not rejected."
    )

print("=" * 60)
print("Invalid Scoring Output testing: PASS")

missing_risk_score: REJECTED
invalid_risk_score: REJECTED
invalid_ml_score: REJECTED
invalid_rule_flags: REJECTED
invalid_decision: REJECTED
invalid_latency: REJECTED
invalid_user_id: REJECTED
Invalid Scoring Output testing: PASS


In [11]:
def mock_person_a_producer():

    return {
        "transaction_id": "MOCK_A_000001",
        "user_id": "MOCK_USER_000001",
        "amount": 175.0,
        "amount_vs_avg_ratio": 1.2,
        "txn_count_last_5min": 1,
        "time_since_last_txn_sec": 240.0,
        "distance_from_last_location_km": 4.0,
        "merchant_category_is_new_for_user": 0
    }

person_a_output = mock_person_a_producer()

print(
    json.dumps(
        person_a_output,
        indent=2
    )
)

print("\nMock Person A producer: READY")

{
  "transaction_id": "MOCK_A_000001",
  "user_id": "MOCK_USER_000001",
  "amount": 175.0,
  "amount_vs_avg_ratio": 1.2,
  "txn_count_last_5min": 1,
  "time_since_last_txn_sec": 240.0,
  "distance_from_last_location_km": 4.0,
  "merchant_category_is_new_for_user": 0
}

Mock Person A producer: READY


In [12]:
person_a_errors = validate_feature_vector(
    person_a_output
)

print("PERSON A → PERSON B MOCK HANDOFF")
print("=" * 60)

if person_a_errors:
    for error in person_a_errors:
        print(error)

    raise ValueError(
        "Mock Person A → Person B handoff failed."
    )

print("Feature Vector received: PASS")
print("Contract validation: PASS")

PERSON A → PERSON B MOCK HANDOFF
Feature Vector received: PASS
Contract validation: PASS


In [13]:
def mock_person_b_scoring(feature_data):

    return {
        "transaction_id": feature_data["transaction_id"],
        "user_id": feature_data["user_id"],
        "risk_score": 0.32,
        "rule_flags": [],
        "ml_fraud_score": 0.41,
        "decision": "OTP",
        "human_readable_reason": (
            "Transaction requires additional verification."
        ),
        "processed_at": datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),
        "latency_ms": 5.2
    }

person_b_output = mock_person_b_scoring(
    person_a_output
)

print("MOCK PERSON B OUTPUT")
print("=" * 60)
print(
    json.dumps(
        person_b_output,
        indent=2
    )
)

MOCK PERSON B OUTPUT
{
  "transaction_id": "MOCK_A_000001",
  "user_id": "MOCK_USER_000001",
  "risk_score": 0.32,
  "rule_flags": [],
  "ml_fraud_score": 0.41,
  "decision": "OTP",
  "human_readable_reason": "Transaction requires additional verification.",
  "processed_at": "2026-09-06T00:51:19.388622+00:00",
  "latency_ms": 5.2
}


In [14]:
person_b_errors = validate_scoring_output(
    person_b_output
)

print("PERSON B → PERSON C MOCK HANDOFF")
print("=" * 60)

if person_b_errors:
    for error in person_b_errors:
        print(error)

    raise ValueError(
        "Mock Person B → Person C handoff failed."
    )

print("Scoring Output received: PASS")
print("Contract validation: PASS")

PERSON B → PERSON C MOCK HANDOFF
Scoring Output received: PASS
Contract validation: PASS


In [15]:
mock_transaction = mock_person_a_producer()

input_errors = validate_feature_vector(
    mock_transaction
)

if input_errors:
    raise ValueError(
        "Mock A → B input validation failed."
    )

mock_detection_result = mock_person_b_scoring(
    mock_transaction
)

output_errors = validate_scoring_output(
    mock_detection_result
)

if output_errors:
    raise ValueError(
        "Mock B → C output validation failed."
    )

if (
    mock_transaction["transaction_id"]
    != mock_detection_result["transaction_id"]
):
    raise ValueError(
        "transaction_id was not preserved across the handoff."
    )

if (
    mock_transaction["user_id"]
    != mock_detection_result["user_id"]
):
    raise ValueError(
        "user_id was not preserved across the handoff."
    )

print("FULL MOCK CONTRACT FLOW")
print("=" * 60)
print("Person A → Feature Vector: PASS")
print("Feature Vector → Person B: PASS")
print("Person B → Scoring Output: PASS")
print("Scoring Output → Person C: PASS")
print("Transaction ID preservation: PASS")
print("User ID preservation: PASS")
print("=" * 60)
print("Full mock A → B → C contract flow: PASS")

FULL MOCK CONTRACT FLOW
Person A → Feature Vector: PASS
Feature Vector → Person B: PASS
Person B → Scoring Output: PASS
Scoring Output → Person C: PASS
Transaction ID preservation: PASS
User ID preservation: PASS
Full mock A → B → C contract flow: PASS


In [16]:
decision_test_results = []

for decision in [
    "Allow",
    "OTP",
    "Review",
    "Block"
]:

    test_output = mock_scoring_output.copy()
    test_output["decision"] = decision

    errors = validate_scoring_output(
        test_output
    )

    decision_test_results.append({
        "decision": decision,
        "valid": len(errors) == 0,
        "errors": errors
    })

for result in decision_test_results:
    print(
        f"{result['decision']}: "
        f"{'PASS' if result['valid'] else 'FAIL'}"
    )

failed_decisions = [
    result
    for result in decision_test_results
    if not result["valid"]
]

if failed_decisions:
    raise RuntimeError(
        "One or more decision values failed contract validation."
    )

print("=" * 60)
print("All four decision values: PASS")

Allow: PASS
OTP: PASS
Review: PASS
Block: PASS
All four decision values: PASS


In [17]:
person_a_contract_path = (
    METRICS_DIR /
    "person_a_to_person_b_contract.json"
)

person_b_contract_path = (
    METRICS_DIR /
    "person_b_to_person_c_contract.json"
)

person_a_contract = {
    "contract": "Person A → Person B",
    "name": "Feature Vector",
    "fields": {
        name: (
            ["bool", "int"]
            if name == "merchant_category_is_new_for_user"
            else (
                ["float", "int"]
                if name in {
                    "amount",
                    "amount_vs_avg_ratio",
                    "time_since_last_txn_sec",
                    "distance_from_last_location_km"
                }
                else ["string"]
            )
        )
        for name in PERSON_A_TO_B_FIELD_NAMES
    }
}

person_b_contract = {
    "contract": "Person B → Person C",
    "name": "Scoring Output",
    "fields": {
        "transaction_id": ["string"],
        "user_id": ["string"],
        "risk_score": ["float", "int"],
        "rule_flags": ["array"],
        "ml_fraud_score": ["float", "int"],
        "decision": [
            "Allow",
            "OTP",
            "Review",
            "Block"
        ],
        "human_readable_reason": ["string"],
        "processed_at": ["ISO 8601 string"],
        "latency_ms": ["float", "int"]
    }
}

with open(
    person_a_contract_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        person_a_contract,
        file,
        indent=2
    )

with open(
    person_b_contract_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        person_b_contract,
        file,
        indent=2
    )

print("CONTRACT FILES SAVED")
print("=" * 60)
print(person_a_contract_path)
print(person_b_contract_path)

CONTRACT FILES SAVED
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\person_a_to_person_b_contract.json
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\person_b_to_person_c_contract.json


In [18]:
mock_integration_path = (
    REPORTS_DIR /
    "phase15_mock_integration_examples.json"
)

mock_examples = {
    "person_a_feature_vector": person_a_output,
    "person_b_scoring_output": person_b_output,
    "full_flow": {
        "input": mock_transaction,
        "output": mock_detection_result
    }
}

with open(
    mock_integration_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        mock_examples,
        file,
        indent=2,
        default=str
    )

print("Mock integration examples saved:")
print(mock_integration_path)

Mock integration examples saved:
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\reports\phase15_mock_integration_examples.json


In [19]:
phase15_report = {
    "phase": 15,
    "title": "Integration Contracts",
    "project_version": "V2.1",
    "status": "COMPLETED",
    "contracts": {
        "person_a_to_person_b": {
            "name": "Feature Vector",
            "fields": PERSON_A_TO_B_FIELD_NAMES
        },
        "person_b_to_person_c": {
            "name": "Scoring Output",
            "fields": PERSON_B_TO_C_FIELD_NAMES
        }
    },
    "tests": {
        "valid_feature_vector": True,
        "invalid_feature_vectors": len(
            invalid_feature_vector_results
        ),
        "valid_scoring_output": True,
        "invalid_scoring_outputs": len(
            invalid_scoring_results
        ),
        "mock_person_a_to_b": True,
        "mock_person_b_to_c": True,
        "full_mock_flow": True,
        "all_decisions": True
    },
    "output_files": {
        "person_a_contract": str(
            person_a_contract_path
        ),
        "person_b_contract": str(
            person_b_contract_path
        ),
        "mock_examples": str(
            mock_integration_path
        )
    }
}

phase15_report_path = (
    REPORTS_DIR /
    "phase15_integration_contracts_report.json"
)

with open(
    phase15_report_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase15_report,
        file,
        indent=2,
        default=str
    )

print("Phase 15 report saved:")
print(phase15_report_path)

Phase 15 report saved:
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\reports\phase15_integration_contracts_report.json


In [20]:
final_checks = {
    "project_found": PROJECT_ROOT.exists(),

    "feature_vector_contract_valid": (
        len(
            validate_feature_vector(
                feature_vector
            )
        ) == 0
    ),

    "scoring_output_contract_valid": (
        len(
            validate_scoring_output(
                mock_scoring_output
            )
        ) == 0
    ),

    "invalid_feature_vectors_detected": (
        len(unexpected_acceptance) == 0
    ),

    "invalid_scoring_outputs_detected": (
        len(
            [
                result
                for result in invalid_scoring_results
                if not result["rejected"]
            ]
        ) == 0
    ),

    "person_a_to_b_mock_passed": (
        len(person_a_errors) == 0
    ),

    "person_b_to_c_mock_passed": (
        len(person_b_errors) == 0
    ),

    "full_mock_flow_passed": True,

    "all_decisions_valid": (
        len(failed_decisions) == 0
    ),

    "contract_a_file_saved": (
        person_a_contract_path.exists()
    ),

    "contract_b_file_saved": (
        person_b_contract_path.exists()
    ),

    "mock_examples_saved": (
        mock_integration_path.exists()
    ),

    "phase15_report_saved": (
        phase15_report_path.exists()
    )
}

overall_verification = all(
    final_checks.values()
)

print("PHASE 15 FINAL STATUS")
print("=" * 60)

for name, status in final_checks.items():
    print(f"{name}: {status}")

print("=" * 60)
print(
    f"Overall verification: {overall_verification}"
)

if overall_verification:
    print("PHASE 15 STATUS: READY")
else:
    print("PHASE 15 STATUS: REVIEW REQUIRED")

PHASE 15 FINAL STATUS
project_found: True
feature_vector_contract_valid: True
scoring_output_contract_valid: True
invalid_feature_vectors_detected: True
invalid_scoring_outputs_detected: True
person_a_to_b_mock_passed: True
person_b_to_c_mock_passed: True
full_mock_flow_passed: True
all_decisions_valid: True
contract_a_file_saved: True
contract_b_file_saved: True
mock_examples_saved: True
phase15_report_saved: True
Overall verification: True
PHASE 15 STATUS: READY
